# Bài 2 – Trang 15: Phân lớp Adult Income

**6451071011 - Lê Nguyễn Anh Dự**

Chạy tuần tự các ô trong Google Colab. Nếu hai CSV chưa có trong thư mục làm việc, notebook tự tải từ các liên kết Drive đã cung cấp.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

TRAIN_ID = '1BYKP73pyzphA5YHR1fVvn2A0kBpr09TB'
TEST_ID = '1Vh7e1IBL0SRxjT3gU_v_rnIcAT18griD'

def ensure_csv(filename, file_id):
    if os.path.exists(filename):
        return filename
    import subprocess, sys
    try:
        import gdown
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
        import gdown
    gdown.download(id=file_id, output=filename, quiet=False)
    if not os.path.exists(filename):
        raise FileNotFoundError(f'Không tải được {filename}; hãy tải tệp từ Drive và đưa lên Colab.')
    return filename

def heading(n, title):
    print('6451071011 - Lê Nguyễn Anh Dự')
    print(f'Câu {n}. {title}')


## Câu 1. Nhập dữ liệu và đếm số dòng

In [ ]:
heading(1, 'Số lượng dữ liệu ban đầu')
train_raw = pd.read_csv(ensure_csv('adult.data.csv', TRAIN_ID), skipinitialspace=True)
test_raw = pd.read_csv(ensure_csv('adult.test.csv', TEST_ID), skipinitialspace=True)
train_raw.columns = train_raw.columns.str.strip().str.lower()
test_raw.columns = test_raw.columns.str.strip().str.lower()
print(f'Huấn luyện: {len(train_raw):,} dòng, {train_raw.shape[1]} cột')
print(f'Kiểm thử: {len(test_raw):,} dòng, {test_raw.shape[1]} cột')
display(train_raw.head(3), test_raw.head(3))


## Câu 2. Tiền xử lý và nối dữ liệu

In [ ]:
heading(2, 'Xóa dòng có ?, bỏ fnlwgt và nối dữ liệu')
def clean(df):
    df = df.copy()
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.strip()
    df = df.replace(r'^\?$', np.nan, regex=True).dropna().reset_index(drop=True)
    return df

train_clean = clean(train_raw).drop(columns='fnlwgt')
test_clean = clean(test_raw)
assert list(train_clean.columns) == list(test_clean.columns), 'Hai tệp có cột không khớp.'
train_count = len(train_clean)
test_count = len(test_clean)
data = pd.concat([train_clean, test_clean], ignore_index=True)
print(f'Sau làm sạch: train={train_count:,}, test={test_count:,}; gộp={len(data):,} dòng, {data.shape[1]} cột')
display(data.head(), data.info())


## Câu 3. Khảo sát tương quan Pearson

Pearson áp dụng cho các cột số. Chỉ xét các thuộc tính đầu vào; không dùng nhãn `income` để quyết định bỏ cột. Ngưỡng tương quan cao: |r| ≥ 0,90.

In [ ]:
heading(3, 'Ma trận tương quan Pearson')
numeric_features = data.drop(columns='income').select_dtypes(include=np.number)
corr = numeric_features.corr(method='pearson')
display(corr.round(3))
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Tương quan Pearson giữa các thuộc tính số')
plt.tight_layout()
plt.show()
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if (upper[c] >= 0.90).any()]
print('Cột bỏ do tương quan cao:', drop_corr if drop_corr else 'Không có')
data = data.drop(columns=drop_corr)


## Câu 4. Tách thuộc tính đầu vào và nhãn quyết định

In [ ]:
heading(4, 'Tách features và labels')
features = data.drop(columns='income')
labels = data['income'].str.rstrip('.')
print('Kích thước features:', features.shape)
print('Phân bố labels:')
display(labels.value_counts())


## Câu 5. One-hot encoding các cột dạng chữ

In [ ]:
heading(5, 'Chuyển dữ liệu phân loại sang one-hot')
categorical_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()
print('Cột cần chuyển:', categorical_cols)
features_encoded = pd.get_dummies(features, columns=categorical_cols, dtype=np.uint8)
print('Kích thước sau chuyển:', features_encoded.shape)
display(features_encoded.head())


## Câu 6. Tách lại tập huấn luyện và kiểm thử

In [ ]:
heading(6, 'Tách theo số dòng đã làm sạch')
X_train = features_encoded.iloc[:train_count].copy()
X_test = features_encoded.iloc[train_count:].copy()
y_train = labels.iloc[:train_count].copy()
y_test = labels.iloc[train_count:].copy()
assert len(X_test) == test_count and len(X_train) == train_count
assert list(X_train.columns) == list(X_test.columns)
print('X_train:', X_train.shape, 'X_test:', X_test.shape)
print('y_train:', y_train.shape, 'y_test:', y_test.shape)


## Hàm đánh giá dùng chung

Cây ID3 trong bài này được xấp xỉ bằng `DecisionTreeClassifier(criterion="entropy")` như hướng dẫn. Với thuộc tính số, scikit-learn tìm ngưỡng chia, nên đây không phải cài đặt ID3 thuần túy.

In [ ]:
class_names = ['<=50K', '>50K']
results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=class_names)
    print(classification_report(y_test, pred, labels=class_names, zero_division=0))
    print('Ma trận nhầm lẫn (hàng: thực tế; cột: dự đoán):')
    print(cm)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Dự đoán'); plt.ylabel('Thực tế'); plt.title(name)
    plt.tight_layout(); plt.show()
    results.append({'Mô hình': name,
                    'Accuracy': accuracy_score(y_test, pred),
                    'Precision (>50K)': precision_score(y_test, pred, pos_label='>50K', zero_division=0),
                    'Recall (>50K)': recall_score(y_test, pred, pos_label='>50K', zero_division=0),
                    'F1 (>50K)': f1_score(y_test, pred, pos_label='>50K', zero_division=0)})

def show_tree(model, name, filename):
    print('Hình cây hiển thị và lưu 4 tầng đầu để dễ đọc; mô hình vẫn được huấn luyện trên cây đầy đủ.')
    print('Bốn tầng đầu (để đọc trực tiếp trong output):')
    print(export_text(model, feature_names=list(X_train.columns), max_depth=3))
    # Cây đầy đủ có thể rất lớn; ảnh minh họa chỉ vẽ 3 tầng đầu.
    plt.figure(figsize=(22, 10))
    plot_tree(model, feature_names=X_train.columns, class_names=class_names,
              filled=True, rounded=True, max_depth=3, fontsize=8)
    plt.title(f'{name} – 4 tầng đầu')
    plt.tight_layout()
    plt.savefig(filename, dpi=100, bbox_inches='tight')
    plt.show()
    print('Đã lưu:', filename)


## Câu 7. Cây ID3 (entropy)

In [ ]:
heading(7, 'Cây ID3/entropy và ma trận nhầm lẫn')
id3 = DecisionTreeClassifier(criterion='entropy', random_state=42)
id3.fit(X_train, y_train)
evaluate('ID3 (entropy)', id3)
show_tree(id3, 'ID3 (entropy)', 'decision_tree_id3.png')


## Câu 8. Cây CART (Gini)

In [ ]:
heading(8, 'Cây CART/Gini và ma trận nhầm lẫn')
cart = DecisionTreeClassifier(criterion='gini', random_state=42)
cart.fit(X_train, y_train)
evaluate('CART (gini)', cart)
show_tree(cart, 'CART (gini)', 'decision_tree_cart.png')


## Câu 9. Naïve Bayes

In [ ]:
heading(9, 'Gaussian Naïve Bayes và ma trận nhầm lẫn')
nb = GaussianNB()
nb.fit(X_train, y_train)
evaluate('Gaussian Naïve Bayes', nb)


## Câu 10. So sánh kết quả

In [ ]:
heading(10, 'So sánh ba mô hình trên cùng tập kiểm thử')
comparison = pd.DataFrame(results).set_index('Mô hình')
display(comparison.style.format('{:.4f}').highlight_max(axis=0, color='#c8e6c9'))
best_acc = comparison['Accuracy'].idxmax()
best_f1 = comparison['F1 (>50K)'].idxmax()
print(f'Accuracy cao nhất: {best_acc} ({comparison.loc[best_acc, "Accuracy"]:.4f}).')
print(f'F1 cho lớp >50K cao nhất: {best_f1} ({comparison.loc[best_f1, "F1 (>50K)"]:.4f}).')
print('Nhận xét: Accuracy phản ánh tỷ lệ dự đoán đúng chung; F1 và recall cho >50K giúp đánh giá lớp thu nhập cao, vốn ít mẫu hơn. Cây không giới hạn độ sâu có thể quá khớp; GaussianNB giả định các thuộc tính độc lập có điều kiện và phân phối Gaussian.')
